<a href="https://colab.research.google.com/github/Multiomics-Analytics-Group/course_multi-omics_analysis/blob/main/proteomics/notebooks/02_proteomics_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Proteomics Basic Analysis

---

Yesterday we produced a matrix of **1 458 protein groups × 48 runs**. Today we ask it a
question:

> **Which serum proteins distinguish sepsis caused by carbapenem-resistant
> *Klebsiella pneumoniae* (CRKP) from sepsis caused by a susceptible strain (CSKP), or
> from sepsis with no organism found (Con)?**

Getting from a matrix to a defensible answer takes six steps, and each one is a decision
that changes the result:

1. **Transform** — put the intensities on a scale where the statistics make sense.
2. **Filter** — decide which proteins are measured well enough to test.
3. **Impute** — decide what a missing value *means*.
4. **Normalise** — remove differences caused by loading, not biology.
5. **Test** — quantify evidence, and pay the price of testing 1 458 hypotheses.
6. **Interpret** — turn a protein list into biology.

We use [**acore**](https://analytics-core.readthedocs.io/) (analytics core),
[**vuecore**](https://vuecore.readthedocs.io/) and
[**vuegen**](https://vuegen.readthedocs.io/) — the analysis, visualisation and reporting
libraries developed at the NNF BRIGHT / Biosustain Data Science Platform.

In [ ]:
%pip install -q acore vuecore vuegen "pingouin<0.6.0"

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns

COURSE_REPO = "Multiomics-Analytics-Group/course_multi-omics_analysis"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{COURSE_REPO}/{BRANCH}"

# Everything we save here becomes a VueGen report at the end of the notebook.
# VueGen turns directory structure into report structure, so we create the sections now.
OUT_DIR = Path("proteomics/report")
SECTIONS = {
    name: OUT_DIR / name
    for name in ["1_quality_control", "2_differential_abundance", "3_enrichment", "4_data"]
}
for path in SECTIONS.values():
    path.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="deep")
GROUP_COLOURS = {"Con": "#4C72B0", "CSKP": "#DD8452", "CRKP": "#C44E52"}
GROUP_ORDER = ["Con", "CSKP", "CRKP"]

## 1. Load the data

Two files: the protein matrix from the pipeline, and the clinical metadata for the same
45 patients. Keeping them separate — and joining on an explicit sample identifier — is a
habit worth having; encoding group membership in file names is how cohorts get scrambled.

In [ ]:
proteins_wide = pd.read_csv(f"{BASE_URL}/proteomics/data/protein_groups_matrix.tsv", sep="\t")
metadata = pd.read_csv(f"{BASE_URL}/metadata/sample_metadata.tsv", sep="\t").set_index("sample_id")

ANNOTATION_COLS = ["protein_group", "protein_names", "genes", "description"]
protein_info = proteins_wide[ANNOTATION_COLS].set_index("protein_group")

print(f"{proteins_wide.shape[0]} protein groups")
print(f"{metadata.shape[0]} patients: {metadata['group'].value_counts().reindex(GROUP_ORDER).to_dict()}")
protein_info.head(3)

### Orientation matters

Statistics packages — `acore` included — expect **samples in rows, features in columns**,
because a sample is an observation and a protein is a variable. Mass spectrometry software
outputs the transpose. Transposing early avoids a great deal of confusion later.

In [ ]:
patient_ids = [c for c in proteins_wide.columns if c in metadata.index]
data = proteins_wide.set_index("protein_group")[patient_ids].T
data.index.name = "sample"
print(f"data: {data.shape[0]} samples x {data.shape[1]} proteins")
data.iloc[:4, :5]

In [ ]:
data.head()

## 2. Log-transform

MS intensities span orders of magnitude and are **right-skewed and multiplicative**: a
protein that doubles adds a fixed amount on the log scale, not on the linear scale. Almost
every statistical method we use assumes roughly symmetric errors of constant variance,
so we work in log₂ throughout. A log₂ fold-change of 1 means "twice as much".

In [ ]:
data_log = np.log2(data)

# Extract and clean non-NaN values
raw_vals = data.to_numpy().ravel()
raw_vals = raw_vals[~np.isnan(raw_vals)]

log_vals = data_log.to_numpy().ravel()
log_vals = log_vals[~np.isnan(log_vals)]

# Raw intensities plot
fig1 = px.histogram(
    x=raw_vals,
    nbins=80,
    title="Raw intensities",
    labels={"x": "MaxLFQ intensity", "y": "count"},
    color_discrete_sequence=["#4C72B0"]
)
fig1.update_layout(width=500, height=350)
fig1.show()

# Log2 transformation plot
fig2 = px.histogram(
    x=log_vals,
    nbins=80,
    title="After log2 transformation",
    labels={"x": "log2 intensity", "y": "count"},
    color_discrete_sequence=["#4C72B0"]
)
fig2.update_layout(width=500, height=350)
fig2.show()

The right-hand panel is the shape we want. The bump at the low end is characteristic of
label-free proteomics: proteins near the detection limit are measured only when they
happen to be high enough — a bias we must keep in mind when imputing.

## 3. Filter: which proteins can we actually test?

27 % of the matrix is missing. Two questions decide what we keep.

**Is the protein measured often enough?** A t-test on 3 values per group is not worth
running. The paper's rule was: drop proteins missing in more than 70 % of samples, then
drop proteins missing in more than 50 % of samples **in both** groups of a comparison.

**Is the missingness informative?** In proteomics, missing is usually **not at random** —
a protein is absent because it is *low*, not because the instrument looked away. This is
the single biggest difference from, say, transcriptomics, and it drives the choice of
imputation below.

In [ ]:
completeness = data_log.notna().mean()  # fraction of patients per protein

# Create histogram using Plotly Express
fig = px.histogram(
    x=completeness,
    nbins=45,
    title="Completeness distribution",
    labels={
        "x": "Fraction of patients in which the protein was quantified",
        "y": "Protein groups"
    },
    color_discrete_sequence=["#4C72B0"]
)

# Add vertical threshold lines
for cut, colour in [(0.30, "#DD8452"), (0.70, "#C44E52")]:
    fig.add_vline(
        x=cut,
        line_dash="dash",
        line_color=colour,
        annotation_text=f"{cut:.0%} of patients",
        annotation_position="top right"
    )

fig.update_layout(
    width=700,
    height=350,
    showlegend=False,
    margin=dict(t=50, b=50, l=50, r=50)
)
fig.show()

MIN_COMPLETENESS = 0.70
kept = completeness[completeness >= MIN_COMPLETENESS].index
data_filtered = data_log[kept]
print(f"kept {len(kept)} of {data_log.shape[1]} protein groups "
      f"(quantified in >= {MIN_COMPLETENESS:.0%} of patients)")

### ✋ Exercise

Set `MIN_COMPLETENESS` to 0.30 and to 1.00, and re-run from here to the volcano plot.
How many proteins are significant in each case? Which threshold would you defend to a
reviewer, and on what grounds? (There is no single right answer — but "the one that gave
the most hits" is definitely the wrong one.)

## 4. Impute

Because missing values are *low* values, replacing them with the row mean would be
actively misleading. The standard proteomics answer, from MaxQuant/Perseus, is to draw
replacements from a **narrow normal distribution shifted below the observed range** of
each sample: "we did not see it, so assume it was just under the detection limit".

The paper used shift = 1.8 standard deviations, width = 0.25 SD. `acore` implements
exactly this.

> ⚙️ Imputation buys you complete data and costs you honesty about uncertainty. Any
> protein whose "significance" rests mostly on imputed values is a candidate artefact —
> which is why we check that afterwards.

In [ ]:
from acore.imputation_analysis import imputation_normal_distribution

data_imputed = imputation_normal_distribution(data_filtered, shift=1.8, nstd=0.25)
print(f"missing before: {data_filtered.isna().to_numpy().mean():.2%}   "
      f"after: {data_imputed.isna().to_numpy().mean():.2%}")

# Extract and clean values for measured and imputed data
raw_vals = data_filtered.to_numpy().ravel()
raw_vals = raw_vals[~np.isnan(raw_vals)]

imputed_only = data_imputed.where(data_filtered.isna())
imp_vals = imputed_only.to_numpy().ravel()
imp_vals = imp_vals[~np.isnan(imp_vals)]

# Combine into a DataFrame for Plotly Express
df_plot = pd.DataFrame({
    "intensity": np.concatenate([raw_vals, imp_vals]),
    "type": ["measured"] * len(raw_vals) + ["imputed"] * len(imp_vals)
})

# Create overlayed histogram using Plotly Express
fig = px.histogram(
    df_plot,
    x="intensity",
    color="type",
    barmode="overlay",
    nbins=70,
    title="Where the imputed values sit",
    labels={"intensity": "log2 intensity", "count": "count", "type": ""},
    color_discrete_map={"measured": "#4C72B0", "imputed": "#C44E52"}
)

# Apply specific opacities to match the original matplotlib layers
if len(fig.data) >= 2:
    fig.data[0].opacity = 0.75
    fig.data[1].opacity = 0.85

fig.update_layout(
    width=700,
    height=350,
    margin=dict(t=50, b=50, l=50, r=50)
)
fig.show()

The red distribution should sit at the **low end** of the blue one. If it sits in the
middle, something is wrong with the shift parameter and every downstream result is
suspect.

## 5. Normalise

Two samples can differ simply because more peptide was injected. Normalisation removes
that, and only that. **Median normalisation** subtracts each sample's median (on the log
scale) — a robust choice when most proteins do not change, which is the usual situation.

In [ ]:
from acore import normalization

data_norm = normalization.normalize_data(data_imputed, method="median")

# Helper function to reshape wide DataFrames into long format for Plotly Express
def prepare_long_df(frame, metadata):
    df_long = frame.stack().reset_index(name="intensity")
    sample_col = df_long.columns[0]
    df_long.rename(columns={sample_col: "sample"}, inplace=True)
    df_long["group"] = df_long["sample"].map(metadata["group"])
    return df_long

df_imp_long = prepare_long_df(data_imputed, metadata)
df_norm_long = prepare_long_df(data_norm, metadata)

# Plot 1: Before normalisation
fig1 = px.box(
    df_imp_long,
    x="sample",
    y="intensity",
    color="group",
    color_discrete_map=GROUP_COLOURS,
    title="Before normalisation",
    labels={"intensity": "log2 intensity", "sample": ""}
)
fig1.update_layout(
    width=900,
    height=350,
    xaxis={"categoryorder": "array", "categoryarray": list(data_imputed.index)},
    margin=dict(t=50, b=100, l=50, r=50)
)
fig1.update_xaxes(tickangle=90)
fig1.show()

# Plot 2: After median normalisation
fig2 = px.box(
    df_norm_long,
    x="sample",
    y="intensity",
    color="group",
    color_discrete_map=GROUP_COLOURS,
    title="After median normalisation",
    labels={"intensity": "log2 intensity", "sample": ""}
)
fig2.update_layout(
    width=900,
    height=350,
    xaxis={"categoryorder": "array", "categoryarray": list(data_norm.index)},
    margin=dict(t=50, b=100, l=50, r=50)
)
fig2.update_xaxes(tickangle=90)
fig2.show()

Sample labels are coloured by group. After normalisation the medians line up, and — this
is the point — they line up *without* the groups separating, confirming that the
correction is technical rather than biological.

> There are other methods available: `method="quantile"` and `method="zscore"`.

## 6. Do the samples group by biology? — PCA

Before testing single proteins, look at the whole matrix. **Principal component analysis**
rotates the 45 samples into new axes ordered by how much variance they explain. If the
dominant structure in the data corresponds to our groups, we should be able to see it. If
it corresponds to something else — a batch, a hospital, a sex imbalance — we need to know
*before* interpreting p-values.

In [ ]:
from acore.decomposition.pca import run_pca

pcs, pca_model = run_pca(data_norm, n_components=4)
explained = pca_model.explained_variance_ratio_ * 100
pcs = pcs.join(metadata[["group", "sex", "age", "c_reactive_protein", "procalcitonin"]])

pc1_col = pcs.columns[0]
pc2_col = pcs.columns[1]

# Plot 1: Coloured by group
fig1 = px.scatter(
    pcs,
    x=pc1_col,
    y=pc2_col,
    color="group",
    color_discrete_map=GROUP_COLOURS,
    category_orders={"group": GROUP_ORDER},
    title="Coloured by group",
    labels={
        pc1_col: f"PC1 ({explained[0]:.1f}%)",
        pc2_col: f"PC2 ({explained[1]:.1f}%)"
    }
)
fig1.update_traces(marker=dict(size=12, line=dict(width=1, color="white")))
fig1.update_layout(width=600, height=450, margin=dict(t=50, b=50, l=50, r=50))
fig1.show()

# Plot 2: Coloured by age
fig2 = px.scatter(
    pcs,
    x=pc1_col,
    y=pc2_col,
    color="age",
    color_continuous_scale="viridis",
    title="Coloured by age",
    labels={
        pc1_col: f"PC1 ({explained[0]:.1f}%)",
        pc2_col: f"PC2 ({explained[1]:.1f}%)",
        "age": "age (years)"
    }
)
fig2.update_traces(marker=dict(size=12, line=dict(width=1, color="white")))
fig2.update_layout(width=600, height=450, margin=dict(t=50, b=50, l=50, r=50))
fig2.show()

# Optional: To save the plots like the original matplotlib code
# fig1.write_image(SECTIONS["1_quality_control"] / "1_pca_group.png", width=600, height=450, scale=2)
# fig2.write_image(SECTIONS["1_quality_control"] / "1_pca_age.png", width=600, height=450, scale=2)

Do not be disappointed if the groups do not separate. In a cohort of *all septic
patients*, the largest source of variance is how sick each individual is — not which
organism grew. A clean separation here would actually be suspicious. The signal we are
after is in specific proteins, and PCA on all 900-odd proteins is the wrong instrument
for finding it.

### How much variance do we explain, and by what?

In [ ]:
variance_table = pd.DataFrame(
    {"principal component": [f"PC{i + 1}" for i in range(len(explained))],
     "variance explained (%)": explained.round(1),
     "cumulative (%)": explained.cumsum().round(1)}
)
variance_table

### A quick check for confounding

If a clinical variable correlates strongly with PC1, it is competing with our grouping for
the same variance — and if it is also unbalanced between groups, it is a confounder.

In [ ]:
numeric_clinical = ["age", "bmi", "c_reactive_protein", "procalcitonin",
                    "creatinine", "neutrophil_lymphocyte_ratio", "platelet_count", "haemoglobin"]
pc_cols = [c for c in pcs.columns if c.startswith("PC")]
combined = pcs[pc_cols].join(metadata[numeric_clinical])
correlations = combined.corr()

# Create correlation heatmap using Plotly Express
fig = px.imshow(
    correlations,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    zmin=-1,
    zmax=1,
    title="Clinical variables vs principal components",
    labels=dict(color="Pearson r")
)

fig.update_layout(
    width=800,
    height=750,
    margin=dict(t=50, b=50, l=50, r=50)
)
fig.show()

## 7. Differential abundance

Now the central question, one protein at a time. With three groups we do it in two stages:

- **ANOVA** across all three groups: "does this protein differ *anywhere*?"
- **Pairwise tests** for the comparison we care about most, CRKP vs CSKP.

### The multiple-testing problem

We are about to run ~900 tests. At α = 0.05, roughly 45 proteins would look significant
even if nothing were true. The **Benjamini–Hochberg** procedure controls the
**false discovery rate**: it adjusts p-values so that, among the proteins you call
significant, the expected fraction of false calls is at most α. The adjusted value is
reported as `padj`.

In [ ]:
import acore.differential_regulation

analysis_data = data_norm.join(metadata[["group"]])
analysis_data["group"] = pd.Categorical(analysis_data["group"], categories=GROUP_ORDER, ordered=True)
analysis_data["group"] = analysis_data["group"].astype(str)

anova = acore.differential_regulation.run_anova(
    analysis_data,
    alpha=0.05,
    drop_cols=[],
    subject=None,
    group="group",
)
print("columns returned by acore:", list(anova.columns))
anova.head()

`acore` returns slightly different column names depending on the test it chose, so we
normalise them once and use our own names from here on.

In [ ]:
COLUMN_ALIASES = {
    "p-value": "pvalue", "pval": "pvalue", "p_value": "pvalue",
    "Log2FC": "log2FC", "log2_FC": "log2FC",
    "-log10 p-value": "-log10 pvalue",
    "FC": "fold_change",
}


def tidy(results: pd.DataFrame) -> pd.DataFrame:
    """Harmonise acore's result columns and guarantee an 'identifier' column."""
    out = results.rename(columns=COLUMN_ALIASES).copy()
    if "identifier" not in out.columns:
        out = out.rename_axis("identifier").reset_index()
    if "-log10 pvalue" not in out and "pvalue" in out:
        out["-log10 pvalue"] = -np.log10(out["pvalue"])
    if "rejected" in out:
        out["rejected"] = out["rejected"].astype(bool)
    return out


anova = tidy(anova)
significant_anova = anova.query("padj < 0.05") if "padj" in anova else anova.query("pvalue < 0.05")
print(f"proteins with an overall group effect: {len(significant_anova)}")
anova.sort_values("pvalue").head(10)

### CRKP versus CSKP

This is the clinically useful comparison: both groups have a confirmed *K. pneumoniae*
infection, and they differ only in resistance. The paper reports **85 differentially
expressed proteins** here, using an unadjusted p < 0.05 together with
|log₂FC| ≥ 0.58 (a 1.5-fold change).

In [ ]:
pairwise = tidy(
    acore.differential_regulation.run_ttest(
        analysis_data,
        condition1="CRKP",
        condition2="CSKP",
        alpha=0.05,
        drop_cols=[],
        subject=None,
        group="group",
        is_logged=True,
    )
)
print("columns:", list(pairwise.columns))

P_CUTOFF, LFC_CUTOFF = 0.05, 0.58
hits = pairwise[(pairwise["pvalue"] < P_CUTOFF) & (pairwise["log2FC"].abs() >= LFC_CUTOFF)]
print(f"\nCRKP vs CSKP, paper's criteria (p < {P_CUTOFF}, |log2FC| >= {LFC_CUTOFF}): {len(hits)} proteins")
if "padj" in pairwise:
    print(f"surviving FDR correction at 5%: {(pairwise['padj'] < 0.05).sum()} proteins")
pairwise.sort_values("pvalue").head(10)

> ⚙️ Note the gap between the two counts. Applying the paper's criteria gives a
> substantial list; requiring FDR control usually gives far fewer. Neither is wrong — they
> answer different questions. An unadjusted list is a **hypothesis-generating** list that
> must be validated (which the authors did, by targeted PRM in an independent cohort).
> Reporting it as if it were FDR-controlled would not be honest.

### The volcano plot

Effect size on x, evidence on y. The proteins in the upper corners are both large and
well-supported; those high in the middle are well-measured but barely change; those far
out at the bottom are noise in poorly-measured proteins.

In [ ]:
volcano = pairwise.copy()
volcano["gene"] = volcano["identifier"].map(protein_info["genes"])
volcano["significant"] = (volcano["pvalue"] < P_CUTOFF) & (volcano["log2FC"].abs() >= LFC_CUTOFF)

volcano_fig = px.scatter(
    volcano,
    x="log2FC",
    y="-log10 pvalue",
    color="significant",
    hover_data=["identifier", "gene", "pvalue"],
    color_discrete_map={True: "#C44E52", False: "#9AA5B1"},
    title="CRKP vs CSKP — serum proteome",
    labels={"log2FC": "log2 fold change (CRKP / CSKP)", "-log10 pvalue": "-log10 p-value"},
    width=900,
    height=600,
)
volcano_fig.add_hline(y=-np.log10(P_CUTOFF), line_dash="dot", line_color="grey")
volcano_fig.add_vline(x=LFC_CUTOFF, line_dash="dot", line_color="grey")
volcano_fig.add_vline(x=-LFC_CUTOFF, line_dash="dot", line_color="grey")
volcano_fig.write_json(str(SECTIONS["2_differential_abundance"] / "0_volcano_plot.json"))
volcano_fig

## 8. Do we reproduce the paper?

This is the most valuable thing you can do with a public dataset, and it is almost never
done. We re-derived a protein list from the same matrix with our own choices of filtering,
imputation and normalisation. How much of the published list do we recover?

In [ ]:
published = pd.read_csv(f"{BASE_URL}/proteomics/data/published_deps.tsv", sep="\t")
published_crkp_vs_cskp = set(published.query("comparison == 'CRKP_vs_KP'")["protein_group"])
ours = set(hits["identifier"])

overlap = ours & published_crkp_vs_cskp
print(f"published DEPs (CRKP vs CSKP) : {len(published_crkp_vs_cskp)}")
print(f"our DEPs                      : {len(ours)}")
print(f"in both                       : {len(overlap)}")
print(f"only ours                     : {len(ours - published_crkp_vs_cskp)}")
print(f"only published                : {len(published_crkp_vs_cskp - ours)}")

In [ ]:
comparison = published.query("comparison == 'CRKP_vs_KP'").set_index("protein_group")
shared = sorted(overlap)

if shared:
    ours_indexed = hits.set_index("identifier")

    # Create DataFrame for Plotly Express
    df_plot = pd.DataFrame({
        "published": comparison.loc[shared, "log2_fold_change"],
        "ours": ours_indexed.loc[shared, "log2FC"]
    })

    # Create scatter plot
    fig = px.scatter(
        df_plot,
        x="published",
        y="ours",
        title="Effect sizes agree, for the proteins we both find",
        labels={
            "published": "published log2 fold change",
            "ours": "our log2 fold change"
        },
        color_discrete_sequence=["#4C72B0"]
    )

    # Adjust marker appearance (size=45 in matplotlib translates roughly to size=8-9 in plotly)
    fig.update_traces(marker=dict(size=8, opacity=0.75))

    # Add diagonal line (y = x) and 0-intercept reference lines
    lims = [-3, 3]
    fig.add_shape(
        type="line",
        x0=lims[0], y0=lims[0], x1=lims[1], y1=lims[1],
        line=dict(color="grey", dash="dot")
    )
    fig.add_hline(y=0, line=dict(color="grey", width=0.5))
    fig.add_vline(x=0, line=dict(color="grey", width=0.5))

    # Configure layout limits, dimensions, and margins
    fig.update_layout(
        width=520,
        height=500,
        xaxis=dict(range=lims),
        yaxis=dict(range=lims),
        margin=dict(t=50, b=50, l=50, r=50)
    )
    fig.show()

🧬 **Read this carefully, because it is the honest lesson of the day.** The effect sizes
agree closely — the biology is in the data. But the *lists* differ, because a
differential-abundance list is not a property of the data alone; it is a property of the
data **plus the analyst's choices**. Different filtering and imputation, same matrix,
different list.

This is why methods sections matter, why sharing raw data matters, and why a single
unvalidated protein list should never be the end of a study.

## 9. From proteins to biology: functional enrichment

A list of 80 accessions is not an answer. Enrichment analysis asks whether our list
contains more members of some functional category than chance would predict — testing,
for each Gene Ontology term, a 2×2 table with Fisher's exact test.

Two choices decide whether the result means anything:

- **The background.** It must be the proteins we *could have detected* — our filtered
  matrix — not the whole human proteome. A serum proteome is already a biased slice, and
  using the wrong background reports that bias as a discovery.
- **The annotation source.** GO biological process, GO molecular function, KEGG pathways
  and Reactome will not agree with each other, and none of them is complete.

In [ ]:
from acore.io.uniprot import fetch_annotations, process_annotations

annotation_dir = OUT_DIR / "3_enrichment"
annotation_dir.mkdir(parents=True, exist_ok=True)
annotation_file = annotation_dir / "uniprot_annotations.csv"

# Multi-protein groups such as 'A0A075B6P5;P01615' are queried by their leading accession.
accessions = pd.Index(sorted({p.split(";")[0] for p in data_norm.columns}))
FIELDS = "accession,go_p,go_f"

if annotation_file.exists():
    annotations = pd.read_csv(annotation_file, index_col=0)
    print(f"loaded cached annotations for {annotations['identifier'].nunique()} proteins")
else:
    try:
        raw = fetch_annotations(accessions, fields=FIELDS)
        annotations = process_annotations(raw, fields=FIELDS)
        annotations.to_csv(annotation_file)
        print(f"fetched {len(annotations)} annotation rows from UniProt")
    except Exception as error:  # offline, rate-limited, API change ...
        annotations = None
        print("Could not reach the UniProt API:", error)

if annotations is not None:
    display(annotations.head())

The annotation table is *long*: one row per (protein, term) pair, because a protein
belongs to many processes. This shape is what enrichment functions expect.

In [ ]:
if annotations is not None:
    regulation = pairwise.copy()
    regulation["identifier"] = regulation["identifier"].str.split(";").str[0]

    enriched = acore.enrichment_analysis.run_up_down_regulation_enrichment(
        regulation_data=regulation,
        annotation=annotations,
        identifier="identifier",
        annotation_col="annotation" if "annotation" in annotations.columns else annotations.columns[-1],
        pval_col="pvalue",
        lfc_cutoff=LFC_CUTOFF,
        min_detected_in_set=3,
        correction_alpha=0.25,
    )
    print(f"{len(enriched)} enriched terms")
    display(enriched.head(15))

In [ ]:
if annotations is not None and len(enriched):
    p_col = "padj" if "padj" in enriched.columns else "pvalue"
    top = enriched.nsmallest(15, p_col).copy()
    label_col = "terms" if "terms" in top.columns else top.columns[0]

    # Truncate labels and compute -log10 p-value
    top["truncated_label"] = top[label_col].astype(str).str.slice(0, 60)
    top["neg_log10_p"] = -np.log10(top[p_col])

    # Sort ascending=False to match matplotlib's vertical orientation (most significant at top)
    top = top.sort_values(p_col, ascending=False)

    fig = px.bar(
        top,
        x="neg_log10_p",
        y="truncated_label",
        orientation="h",
        title="Enriched functional terms, CRKP vs CSKP",
        labels={
            "neg_log10_p": "-log10 adjusted p-value" if p_col == "padj" else "-log10 p-value",
            "truncated_label": ""
        },
        color_discrete_sequence=["#4C72B0"]
    )

    fig.update_layout(
        width=900,
        height=500,
        margin=dict(t=50, b=50, l=150, r=50)
    )
    fig.show()

### What the paper found

For reference, the published KEGG enrichment of the CRKP-vs-CSKP proteins. Comparing your
terms with theirs is a good test of whether your interpretation is robust to
analytical choices.

In [ ]:
published_kegg = pd.read_csv(f"{BASE_URL}/proteomics/data/published_kegg_deps.tsv", sep="\t")
(published_kegg.query("comparison == 'CRKP_vs_KP'")
 .nsmallest(12, "p_value")[["description", "gene_ratio", "fold_enrichment", "p_value", "genes"]])

## 10. The candidate biomarkers

The paper's four-protein diagnostic panel — **IGHV1-8, ITGA2, PKP1, IGFBP6** — reached
AUC 0.920 for CRKP vs CSKP, and **CD44** and **B2M** were the hub proteins of the
interaction network. Let us look at them one by one, which is what you should always do
before believing a panel.

In [ ]:
PANEL = ["IGHV1-8", "ITGA2", "PKP1", "IGFBP6"]
HUBS = ["CD44", "B2M", "MAT2B", "VWF"]

gene_to_group = (
    protein_info.reset_index()
    .dropna(subset=["genes"])
    .assign(gene=lambda d: d["genes"].str.split(";").str[0])
    .drop_duplicates("gene")
    .set_index("gene")["protein_group"]
)

available = {g: gene_to_group[g] for g in PANEL + HUBS
             if g in gene_to_group.index and gene_to_group[g] in data_norm.columns}
print("found in the filtered matrix:", list(available))
missing = [g for g in PANEL + HUBS if g not in available]
if missing:
    print("not available after filtering:", missing)

In [ ]:
if available:
    tidy_long = (
        data_norm[list(available.values())]
        .rename(columns={v: k for k, v in available.items()})
        .join(metadata[["group"]])
        .melt(id_vars="group", var_name="gene", value_name="log2 intensity")
    )

    fig = px.box(
        tidy_long,
        x="group",
        y="log2 intensity",
        color="group",
        facet_col="gene",
        facet_col_wrap=4,
        points="all",
        color_discrete_map=GROUP_COLOURS,
        category_orders={"group": GROUP_ORDER},
        title="Candidate biomarkers across the severity axis",
        labels={"log2 intensity": "log2 intensity", "group": ""}
    )

    # Remove the forced black color override so box lines and markers
    # correctly inherit the GROUP_COLOURS mapping
    fig.update_traces(
        marker=dict(size=3, opacity=0.5),
        selector=dict(type="box")
    )

    fig.update_layout(
        width=1000,
        height=700,
        showlegend=False,
        margin=dict(t=80, b=50, l=50, r=50)
    )

    fig.show()

⚠️ Notice which genes are *missing* from the plot. Some of the paper's panel proteins did
not survive our completeness filter — they were quantified in too few patients. That is
not a contradiction: the authors filtered differently, so their candidate set included
proteins ours excludes. A biomarker that is only measurable in half your samples is a
fragile biomarker, and this is the kind of thing you should check before designing a
validation assay.

🧬 What to look for: a useful biomarker needs **separation**, not just a significant
p-value. Overlapping boxes with a significant test mean a real but small average
difference — informative about biology, useless for diagnosing an individual patient. That
distinction is the difference between a paper and a clinical test.

### The severity axis

The paper also reports a cluster of 51 proteins that increase **stepwise** from Con to
CSKP to CRKP. That monotone pattern is more convincing than a single pairwise difference,
because it is harder to produce by chance. Let us find such proteins ourselves using
Spearman correlation with the ordered group (Con = 0, CSKP = 1, CRKP = 2).

In [ ]:
from scipy import stats

order_values = metadata.loc[data_norm.index, "group_order"]
trend = pd.DataFrame(
    [
        dict(zip(["rho", "pvalue"], stats.spearmanr(data_norm[protein], order_values)))
        for protein in data_norm.columns
    ],
    index=data_norm.columns,
).rename_axis("protein_group")
trend["gene"] = protein_info["genes"]
trend["direction"] = np.where(trend["rho"] > 0, "increases with resistance", "decreases with resistance")

monotone = trend[(trend["pvalue"] < 0.01) & (trend["rho"].abs() > 0.4)].sort_values("pvalue")
print(f"{len(monotone)} proteins with a monotone trend across Con -> CSKP -> CRKP")
monotone.head(12)

In [ ]:
if len(monotone):
    top_trend = monotone.head(6).index
    tidy_long = (
        data_norm[top_trend]
        .join(metadata[["group"]])
        .melt(id_vars="group", var_name="protein_group", value_name="log2 intensity")
        .merge(protein_info["genes"].rename("gene"), left_on="protein_group", right_index=True, how="left")
    )
    tidy_long["label"] = tidy_long["gene"].fillna(tidy_long["protein_group"])

    # Aggregate to calculate mean and standard error (se) for each group and protein label
    agg_df = (
        tidy_long.groupby(["label", "group"])["log2 intensity"]
        .agg(mean="mean", sem=lambda x: x.sem() if len(x) > 1 else 0)
        .reset_index()
    )

    fig = px.line(
        agg_df,
        x="group",
        y="mean",
        error_y="sem",
        facet_col="label",
        facet_col_wrap=3,
        category_orders={"group": GROUP_ORDER},
        markers=True,
        title="Proteins tracking the Con -> CSKP -> CRKP axis",
        labels={"mean": "log2 intensity", "group": ""}
    )

    # Style lines and markers to match the original #4C72B0 color
    fig.update_traces(
        line=dict(color="#4C72B0", width=2),
        marker=dict(color="#4C72B0", size=8)
    )

    fig.update_layout(
        width=900,
        height=600,
        margin=dict(t=80, b=50, l=50, r=50)
    )
    fig.show()

## 11. Do the proteins relate to the clinic?

A biomarker is more believable if it behaves like the clinical markers we already trust.
**CRP** and **procalcitonin (PCT)** are routine measures of infection severity; PCT in
particular rises in bacterial sepsis. If a candidate protein correlates with them, it is
at least measuring something real.

In [ ]:
clinical = metadata.loc[data_norm.index, ["c_reactive_protein", "procalcitonin",
                                          "neutrophil_lymphocyte_ratio", "creatinine"]]
candidates = list(available.values()) + list(monotone.head(8).index)
candidates = list(dict.fromkeys(candidates))

corr_matrix = pd.DataFrame(
    {
        clinical_var: [
            stats.spearmanr(data_norm[protein], clinical[clinical_var], nan_policy="omit").statistic
            for protein in candidates
        ]
        for clinical_var in clinical.columns
    },
    index=[protein_info.loc[p, "genes"] if pd.notna(protein_info.loc[p, "genes"]) else p
           for p in candidates],
)

# Convert matplotlib's inch-based dynamic height to an approximate pixel height
dynamic_height = int((0.35 * len(corr_matrix) + 2) * 80)

fig = px.imshow(
    corr_matrix,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    zmin=-1,
    zmax=1,
    title="Candidate proteins vs clinical severity markers",
    labels=dict(color="Spearman rho")
)

fig.update_layout(
    width=700,
    height=max(400, dynamic_height),
    margin=dict(t=50, b=50, l=100, r=50)
)
fig.show()

## 12. Save the results and build a report

[**VueGen**](https://vuegen.readthedocs.io/) turns a folder of figures and tables into a
navigable report — no HTML written by hand. The rule is simple: **directory structure
becomes report structure**. Numbered folders become sections, numbered files become
subsections, and `.png` / `.csv` / plotly `.json` files are embedded appropriately.

In [ ]:
data_norm.to_csv(SECTIONS["4_data"] / "1_normalised_protein_matrix.csv")
anova.to_csv(SECTIONS["2_differential_abundance"] / "1_anova_three_groups.csv", index=False)
pairwise.to_csv(SECTIONS["2_differential_abundance"] / "2_ttest_CRKP_vs_CSKP.csv", index=False)
hits.to_csv(SECTIONS["2_differential_abundance"] / "3_significant_proteins.csv", index=False)
trend.to_csv(SECTIONS["2_differential_abundance"] / "4_severity_trend.csv")

!find proteomics/report -type f | sort

Now generate the report. `--report_type html` produces a single self-contained file;
`streamlit` gives an interactive app you can run locally.

In [ ]:
!vuegen --directory proteomics/report --report_type html

## 📚 Further reading

- acore documentation and recipes — <https://analytics-core.readthedocs.io/>
- Tyanova *et al.* (2016) *The Perseus computational platform for comprehensive analysis
  of (prote)omics data.* Nat Methods 13:731–740. — where the imputation strategy comes from.
- Benjamini & Hochberg (1995) *Controlling the false discovery rate.* J R Stat Soc B 57:289–300.
- Timmons *et al.* (2015) *Multiple sources of bias confound functional enrichment
  analysis of global -omics data.* Genome Biol 16:186. — read this before your next
  enrichment analysis.